In [2]:
import time
from pprint import pprint
from tqdm import tqdm

#avenieca sdk imports
from avenieca import Signal
from avenieca.producers import Event
from avenieca.api.model import *
from avenieca.api.eca import ECA

#local file importse
from config import *

def prettyprint(res, status):
    try:
        pprint(res.__dict__)
    except:
        print(res)
    print(status)

load_dotenv()
data_path = os.getenv("DATA_PATH")
url = '%s/bitcoin.csv' % data_path

In [3]:
import pandas as pd
data = pd.read_csv(url)

In [4]:
btc_data = data["price"].values
len(btc_data)

3091

### Stream to ECA

In [5]:
btc_broker_config = btc_twin_config.broker_config
btc_event = Event(config=btc_broker_config)

btc_data = data["price"].values
btc_data_train = btc_data[0:3082]
btc_data_test = btc_data[3082:]
print(btc_data_train[-1])

26469.581684007026


In [7]:
for i in tqdm(range(0, len(btc_data_train))):
    btc_signal = Signal(
        state=[float(btc_data_train[i])]
    )
    
    future = btc_event.publish(btc_signal)
    _ = future.get(timeout=60)
    
    time.sleep(0.2)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3082/3082 [10:57<00:00,  4.69it/s]


In [8]:
index = 1
btc_signal = Signal(
    state=[float(btc_data_test[index])]
)
future = btc_event.publish(btc_signal)
_ = future.get(timeout=60)

### Next state predictions

In [9]:
eca_server = os.getenv("ECA_SERVER")

config = Config(uri=eca_server, username=username, password=password)
eca = ECA(config)


In [10]:
res, status = eca.ess.search(data=Search(
    module_id="btc",
    state=[26469.581684007026],
    limit=1
))
prettyprint(res, status)

[SearchResult(score=0.0, ess=ESSResponse(id=3083, state=[26469.582], module_id='btc', valence=50.0, created_at='2023-08-09T19:48:19.232732Z', updated_at='2023-08-09T19:48:19.232732Z', avg_ess_valence=0.0, total_ess_score=0, avg_ess_score=0, score=0, embedding_input=None, aggregate_id=[], aggregate_valence=[], aggregate_score=[], aggregate_module_id=[], aggregate_shape=[], aggregate_context=[], aggregate_emb_inp=[], context=None))]
200


In [13]:
res, status = eca.ess.get_one_sequence(module_id="btc", sequence_id=-1)
prettyprint(res, status)

{'aggregate_context': [],
 'aggregate_emb_inp': [],
 'aggregate_id': [],
 'aggregate_module_id': [],
 'aggregate_score': [],
 'aggregate_shape': [],
 'aggregate_valence': [],
 'avg_ess_score': 0,
 'avg_ess_valence': 0.0,
 'context': None,
 'created_at': '2023-08-09T19:48:19.232732Z',
 'embedding_input': None,
 'id': 3083,
 'module_id': 'btc',
 'score': 0,
 'state': [26469.582],
 'total_ess_score': 0,
 'updated_at': '2023-08-09T19:48:19.232732Z',
 'valence': 50.0}
200


In [18]:
nsr = NextStateRequest(
    module_id="btc",
    recall=10,
    range=30,
    n=10,
    status='n',
)
res, status = eca.cortex.predictions_raw(data=nsr)
prettyprint(res, status)

{'current_state': [{'aggregate_id': 0,
                    'ess_id': 3083,
                    'module_id': 'btc',
                    'state': [26469.582]}],
 'next_state': [{'list': [{'aggregate_id': 0,
                           'ess_id': 3068,
                           'module_id': 'btc',
                           'state': [26475.607]}]},
                {'list': [{'aggregate_id': 0,
                           'ess_id': 2188,
                           'module_id': 'btc',
                           'state': [26476.13]}]},
                {'list': [{'aggregate_id': 0,
                           'ess_id': 3068,
                           'module_id': 'btc',
                           'state': [26475.607]}]},
                {'list': [{'aggregate_id': 0,
                           'ess_id': 2188,
                           'module_id': 'btc',
                           'state': [26476.13]}]},
                {'list': [{'aggregate_id': 0,
                           'ess_id': 3068,
  

In [12]:
nsr = NextStateRequest(
    module_id="btc",
    recall=3,
    range=3,
    n=10,
    status='n',
)
res, status = eca.cortex.predictions_raw(data=nsr)
prettyprint(res, status)

{'current_state': [{'aggregate_id': 0,
                    'ess_id': 3083,
                    'module_id': 'btc',
                    'state': [26469.582]}],
 'next_state': [{'list': [{'aggregate_id': 0,
                           'ess_id': 3068,
                           'module_id': 'btc',
                           'state': [26475.607]}]},
                {'list': [{'aggregate_id': 0,
                           'ess_id': 2188,
                           'module_id': 'btc',
                           'state': [26476.13]}]},
                {'list': [{'aggregate_id': 0,
                           'ess_id': 3068,
                           'module_id': 'btc',
                           'state': [26475.607]}]},
                {'list': [{'aggregate_id': 0,
                           'ess_id': 2188,
                           'module_id': 'btc',
                           'state': [26476.13]}]},
                {'list': [{'aggregate_id': 0,
                           'ess_id': 3068,
  